# Equities Data Explorer

This notebook explores and prepares the unified `all_stocks` dataframe for ML.

Key inputs and helpers:
- `finance_ml.notebook_config.NotebookConfig` for feature flags
- Database schema and import via `create_equities_schema.sql` and `import_equities_data.sql`
- Optional importer: `load_equities_data.py`

Outputs and steps:
- Robust data loading from DB with CSV fallback
- Preprocessing (numeric/categorical detection, missing values strategy)
- EDA summaries and plots
- Feature engineering aligned with the project’s guidelines


In [1]:
import json
import math
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# Add compatibility layer for NumPy _ARRAY_API attribute
if not hasattr(np, '_ARRAY_API'):
    # Create a mock _ARRAY_API object to prevent AttributeError
    class MockArrayAPI:
        pass


    np._ARRAY_API = MockArrayAPI()
    print("Added compatibility layer for NumPy _ARRAY_API")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML - Import with error handling
try:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, RobustScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer

    print("scikit-learn imports successful")
except AttributeError as e:
    print(f"scikit-learn compatibility issue detected: {e}")
    # Re-add the mock if it was somehow removed
    if not hasattr(np, '_ARRAY_API'):
        class MockArrayAPI:
            pass


        np._ARRAY_API = MockArrayAPI()
    # Retry imports
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, RobustScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer

    print("scikit-learn imports successful after compatibility fix")

# Optional DB libraries
try:
    from sqlalchemy import create_engine, text

    HAVE_SQLALCHEMY = True
except Exception:
    HAVE_SQLALCHEMY = False

# Project root handling (assumes notebook is at repository root)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SQL_SCHEMA = PROJECT_ROOT / "create_equities_schema.sql"
SQL_IMPORT = PROJECT_ROOT / "import_equities_data.sql"

# Optional: ensure finance_ml is importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import sys

print("Python version:", sys.version)

try:
    import numpy as np

    print("NumPy version:", np.__version__)
    print("NumPy location:", np.__file__)
    print("Has _ARRAY_API:", hasattr(np, '_ARRAY_API'))
except Exception as e:
    print("NumPy error:", e)

try:
    import sklearn

    print("scikit-learn version:", sklearn.__version__)
except Exception as e:
    print("scikit-learn error:", e)

from finance_ml.notebook_config import NotebookConfig

CFG = NotebookConfig(
        have_finance_prediction=True,
        have_database_connection=False,  # will detect below
        have_advanced_analytics=True,
        have_dim_reduction=False,
        debug_mode=False,
        )

print("Notebook paths:")
print("  PROJECT_ROOT:", PROJECT_ROOT)
print("  DATA_DIR:    ", DATA_DIR)
print("  SQL_SCHEMA:  ", SQL_SCHEMA)
print("  SQL_IMPORT:  ", SQL_IMPORT)

CFG.display_summary()



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\markm\anaconda3\envs\Finance_ML_Analytics_package\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\markm\anaconda3\envs\Finance_ML_Analytics_package\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\markm\anaconda3\envs\Finance_ML_Analytics_package\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\markm\anaconda3\envs\Finance_ML_Analytics_package\Lib\sit

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

We support both DB and CSV sources:
- DB URL via env var `DB_URL` or explicit string (psycopg2 + SQLAlchemy required)
- CSV fallback reads the four regional CSVs in `data/` and unifies them


In [ ]:
# Configure DB URL (PostgreSQL). Prefer environment variable.
# Example: postgresql+psycopg2://postgres:<pass>@localhost:5432/postgres
DB_URL = os.getenv("DB_URL")

# Detect availability
have_db_url = DB_URL is not None and len(DB_URL) > 0
CFG.have_database_connection = bool(have_db_url and HAVE_SQLALCHEMY)
print("DB available via SQLAlchemy:", CFG.have_database_connection)
if have_db_url:
    print("DB_URL:", DB_URL)
else:
    print("DB_URL not set — will default to CSV fallback unless a DB engine is provided manually.")


Preferred load path if using PostgreSQL:

1) Create schema/table using `create_equities_schema.sql`
2) Import CSVs into DB using `import_equities_data.sql` (staging, NULL handling)

Alternative: Use `load_equities_data.py` which inserts CSVs directly into `equities`.

Note: On Windows, run these in a Terminal/PowerShell, not in this notebook, to avoid PATH issues:
- psql -h localhost -p 5432 -U postgres -d postgres -f create_equities_schema.sql
- psql -h localhost -p 5432 -U postgres -d postgres -f import_equities_data.sql

Or in Python, you can `import load_equities_data` and call its `main()`, but ensure DB credentials are correct.


In [ ]:
def file_exists(p: Path) -> bool:
    try:
        return p.exists()
    except Exception:
        return False


print("Schema SQL present:", file_exists(SQL_SCHEMA))
print("Import SQL present:", file_exists(SQL_IMPORT))

# Optional: Show how to invoke the CSV->Postgres importer safely (do not execute by default).
try:
    import load_equities_data as _led

    print(
        "load_equities_data.py is importable. To load CSVs into Postgres, run _led.main() with correct DB credentials.")
except Exception as e:
    print("load_equities_data.py not importable or has errors:", e)


In [ ]:
def load_from_db(limit: int | None = None) -> pd.DataFrame:
    if not CFG.have_database_connection:
        raise RuntimeError("DB connection not available. Set DB_URL and ensure SQLAlchemy is installed.")
    engine = create_engine(DB_URL)
    query = 'SELECT * FROM equities'
    if limit:
        query += f" LIMIT {int(limit)}"
    df = pd.read_sql(query, engine)
    return df


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Convert column labels to pythonic form: alnum + underscores, lower-case
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
        .str.strip("_")
        .str.lower()
    )
    return df


def load_from_csv(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    # Robust CSV reader that preserves strings and handles empty fields as NaN later
    files = [
        data_dir / "screening_us.csv",
        data_dir / "screening_eu.csv",
        data_dir / "screening_apac.csv",
        data_dir / "screening_rotw.csv",
        ]
    dfs = []
    for f in files:
        if f.exists():
            part = pd.read_csv(f, dtype=str, encoding="utf-8")
            # Backfill Region column if missing based on filename
            if "Region" in part.columns:
                part.loc[part["Region"].isna() | (part["Region"] == ""), "Region"] = (
                    f.stem.split("_")[-1].upper() if "screening_" in f.name else None
                )
            else:
                inferred = f.stem.split("_")[-1].upper() if "screening_" in f.name else None
                part["Region"] = inferred
            dfs.append(part)
        else:
            print(f"Warning: {f} not found, skipping")
    if not dfs:
        raise FileNotFoundError("No CSVs found in data/. Expected screening_us/eu/apac/rotw.csv")
    df = pd.concat(dfs, axis=0, ignore_index=True)
    return df


def coerce_numeric(df: pd.DataFrame, numeric_like_cols: list[str] | None = None) -> pd.DataFrame:
    df = df.copy()
    if numeric_like_cols is None:
        # Heuristic: try to convert any column that looks numeric after stripping commas and %
        candidates = []
        for c in df.columns:
            sample = df[c].dropna().astype(str).head(50)
            if not len(sample):
                continue
            s = sample.str.replace(",", "", regex=False).str.replace("%", "", regex=False)
            if (s.str.fullmatch(r"-?\d+(\.\d+)?").mean() > 0.6):
                candidates.append(c)
        numeric_like_cols = candidates
    for c in numeric_like_cols:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def build_all_stocks(source: str = "auto", limit: int | None = None) -> pd.DataFrame:
    # source: auto|db|csv
    src = source.lower()
    if src == "auto":
        if CFG.have_database_connection:
            df = load_from_db(limit=limit)
        else:
            df = load_from_csv(DATA_DIR)
    elif src == "db":
        df = load_from_db(limit=limit)
    elif src == "csv":
        df = load_from_csv(DATA_DIR)
    else:
        raise ValueError("source must be one of: auto|db|csv")

    df = normalize_columns(df)

    # Ensure essential columns exist (create if missing)
    must_have = ["ticker", "sector", "last_price", "region"]
    for col in must_have:
        if col not in df.columns:
            df[col] = np.nan

    # Basic filtering: drop rows missing key identifiers or price
    df = df.loc[~df["ticker"].isna() & (df["ticker"].astype(str).str.len() > 0)]
    df = df.loc[~df["sector"].isna() & (df["sector"].astype(str).str.len() > 0)]
    df = df.loc[~df["last_price"].isna()]

    # Attempt numeric coercion for common financial fields if present
    numeric_hint_cols = [
        "last_price", "market_cap", "enterprise_value", "ev", "pe", "p_e",
        "ebitda", "net_debt", "revenue", "gross_margin", "ebitda_margin",
        "price_target", "price_target_median"
        ]
    numeric_like = [c for c in numeric_hint_cols if c in df.columns]
    df = coerce_numeric(df, numeric_like_cols=numeric_like)

    # Deduplicate by ticker+region if present
    if "region" in df.columns:
        df = df.sort_values(by=["ticker"]).drop_duplicates(subset=["ticker", "region"], keep="first")
    else:
        df = df.sort_values(by=["ticker"]).drop_duplicates(subset=["ticker"], keep="first")

    df.reset_index(drop=True, inplace=True)
    return df


In [ ]:
# Choose your source: "auto" tries DB first, else CSV
ALL_STOCKS_LIMIT = None  # e.g., 5000 for testing
all_stocks = build_all_stocks(source="auto", limit=ALL_STOCKS_LIMIT)
print(all_stocks.shape)
all_stocks.head(3)


We separate numeric and categorical columns. Missing value strategy:
- Numeric: impute with median per column
- Categorical: impute with a placeholder ("missing")
- Optional: Winsorize or robust scaling later in the pipeline


In [ ]:
# Column detection
numeric_cols = sorted([c for c in all_stocks.columns if pd.api.types.is_numeric_dtype(all_stocks[c])])
categorical_cols = sorted(list(set(all_stocks.columns) - set(numeric_cols)))

print("Numeric columns (sample):", numeric_cols[:15])
print("Categorical columns (sample):", categorical_cols[:15])

# Basic missingness report
missing_summary = (
    all_stocks.isna().mean().sort_values(ascending=False).to_frame("missing_rate")
)
missing_summary.head(20)


In [ ]:
# Counts by Region and Sector
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
if "region" in all_stocks.columns:
    all_stocks["region"].value_counts(dropna=False).plot(kind="bar", ax=axes[0], title="Rows by Region")
if "sector" in all_stocks.columns:
    all_stocks["sector"].value_counts(dropna=False).head(20).plot(kind="bar", ax=axes[1],
                                                                  title="Rows by Sector (Top 20)")
plt.tight_layout();
plt.show()

# Distributions for key metrics (if present)
metrics = ["last_price", "market_cap", "enterprise_value", "ebitda", "pe", "p_e"]
metrics = [m for m in metrics if m in all_stocks.columns]
if metrics:
    n = len(metrics)
    rows = math.ceil(n / 3)
    fig, axes = plt.subplots(rows, 3, figsize=(16, 4 * rows))
    axes = axes.ravel() if n > 1 else [axes]
    for i, m in enumerate(metrics):
        sns.histplot(all_stocks[m].dropna(), bins=50, ax=axes[i])
        axes[i].set_title(f"Distribution: {m}")
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout();
    plt.show()

# Correlations (numeric subset)
if len(numeric_cols) >= 2:
    corr = all_stocks[numeric_cols].corr(method="pearson").fillna(0.0)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Correlation heatmap (numeric columns)")
    plt.show()


We compute a small set of robust features, with sector-aware placeholders you can expand later:
- Ratios: EV/EBITDA, Net_Debt/EBITDA, P/E, P/B (as available)
- One-hot encoding of sector and region
- Robust scaling for numeric features


In [ ]:
def add_basic_ratios(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Define helper to avoid division by zero
    def safe_div(a, b):
        a = pd.to_numeric(a, errors="coerce")
        b = pd.to_numeric(b, errors="coerce")
        return np.where((b == 0) | pd.isna(b), np.nan, a / b)

    # EV/EBITDA
    ev_cols = [c for c in ["enterprise_value", "ev", "market_cap"] if c in df.columns]
    ev_col = ev_cols[0] if ev_cols else None
    if ev_col and "ebitda" in df.columns:
        df["ev_ebitda"] = safe_div(df[ev_col], df["ebitda"])

        # Net_Debt/EBITDA
    if "net_debt" in df.columns and "ebitda" in df.columns:
        df["net_debt_ebitda"] = safe_div(df["net_debt"], df["ebitda"])

        # P/E (prefer `p_e` else `pe`)
    if "p_e" in df.columns:
        df["pe_ratio"] = pd.to_numeric(df["p_e"], errors="coerce")
    elif "pe" in df.columns:
        df["pe_ratio"] = pd.to_numeric(df["pe"], errors="coerce")

    # P/B (common variants)
    for p_b in ["p_b", "pb"]:
        if p_b in df.columns:
            df["pb_ratio"] = pd.to_numeric(df[p_b], errors="coerce")
            break

    # Margins
    for m in ["gross_margin", "ebitda_margin", "operating_margin", "net_margin"]:
        if m in df.columns:
            df[m] = pd.to_numeric(df[m], errors="coerce")

    return df


all_stocks = add_basic_ratios(all_stocks)

# Recompute numeric/cat columns after adding engineered features
numeric_cols = sorted([c for c in all_stocks.columns if pd.api.types.is_numeric_dtype(all_stocks[c])])
categorical_cols = sorted(list(set(all_stocks.columns) - set(numeric_cols)))

print("Engineered columns present:",
      [c for c in ["ev_ebitda", "net_debt_ebitda", "pe_ratio", "pb_ratio"] if c in all_stocks.columns])


In [ ]:
# Example target (if available). Replace with your project target, e.g., price_target.
TARGET_CANDIDATES = ["price_target", "price_target_median"]
TARGET = next((t for t in TARGET_CANDIDATES if t in all_stocks.columns), None)
if TARGET is None:
    print("No price target columns found; creating a dummy target for pipeline illustration")
    TARGET = "dummy_target"
    all_stocks[TARGET] = pd.to_numeric(all_stocks.get("last_price", pd.Series([np.nan] * len(all_stocks))),
                                       errors="coerce")

# Split
X = all_stocks.drop(columns=[TARGET])
y = pd.to_numeric(all_stocks[TARGET], errors="coerce")
mask = ~y.isna()
X, y = X.loc[mask], y.loc[mask]

# Minimal column subsets for the demo
num_subset = [c for c in ["last_price", "market_cap", "ebitda", "ev_ebitda", "net_debt_ebitda", "pe_ratio", "pb_ratio"]
              if c in X.columns]
cat_subset = [c for c in ["sector", "region", "industry", "trading_country", "exchange"] if c in X.columns]

numeric_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", RobustScaler(with_centering=True, with_scaling=True))
    ])

categorical_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_subset),
            ("cat", categorical_transformer, cat_subset),
            ],
        remainder="drop"
        )

# Fit-transform the preprocessor for demonstration
X_small = X[num_subset + cat_subset].copy() if (num_subset or cat_subset) else X.copy()
Xt = preprocessor.fit_transform(X_small)
print("Transformed shape:", Xt.shape)


In [ ]:
checks = {}
checks["has_rows"] = len(all_stocks) > 0
checks["has_ticker"] = "ticker" in all_stocks.columns
checks["has_sector"] = "sector" in all_stocks.columns
checks["has_last_price"] = "last_price" in all_stocks.columns
checks["no_all_nan_last_price"] = all_stocks[
    "last_price"].notna().any() if "last_price" in all_stocks.columns else False
checks["pipeline_ok"] = Xt is not None and isinstance(Xt, np.ndarray) and Xt.shape[0] == X_small.shape[0]

print(json.dumps(checks, indent=2))
assert all(checks.values()), "One or more validation checks failed. Inspect the data source and columns."
print("All validation checks passed.")


Where to integrate project modules:
- `finance_ml` feature functions: replace `add_basic_ratios` with your package’s canonical feature builders.
- Reference: tests `tests/test_features.py`, `tests/test_build_features.py`, `tests/test_preprocess_and_training.py` indicate expected APIs and transformations.
- Keep configuration via env vars and pass-through CLI-equivalent flags to functions as needed.

Suggested improvements:
- Sector-specific pipelines (e.g., separate scalers and encodings per sector)
- Grouped CV by ticker or sector
- SHAP analysis and model ensembling (LightGBM/CatBoost/XGBoost) following the project’s guideline
